In [1]:
from datasets import load_dataset, load_from_disk

In [4]:
dataset = load_dataset(
    "maya-research/IndicVault",
    "Telugu",
    split="train",
)

In [5]:
dataset[0]

{'question': 'ఇండియాలో గ్రోసరీస్ మీద డబ్బులు సేవ్ చేయడానికి బెస్ట్ వేస్ ఏంటి?\n',
 'response': 'ఇండియాలో గ్రోసరీస్ మీద డబ్బులు సేవ్ చేయడానికి చాలా బెస్ట్ వేస్ ఉన్నాయి. ఇక్కడ కొన్ని టిప్స్ ఉన్నాయి చూడండి:\n\n1. **ముందుగానే ప్లాన్ చేసుకోండి:** షాపింగ్ కి వెళ్ళేముందు ఏం కావాలో లిస్ట్ ప్రిపేర్ చేసుకోండి. లిస్ట్ ఉంటే, అనవసరమైనవి కొనకుండా ఉంటారు, డబ్బులు సేవ్ అవుతాయి.\n\n2. **ప్రైస్ కంపేర్ చేయండి:**  ఒకే ప్రోడక్ట్ వేర్వేరు షాపుల్లో వేర్వేరు రేట్లు ఉండొచ్చు. సూపర్\u200cమార్కెట్స్, లోకల్ షాప్స్, ఆన్\u200cలైన్ స్టోర్స్ లో ప్రైస్ కంపేర్ చేసి ఎక్కడ తక్కువ ఉంటే అక్కడ కొనండి.\n\n3. **సేల్స్ అండ్ డిస్కౌంట్స్ చూడండి:**  సూపర్\u200cమార్కెట్స్ లో వీక్లీ సేల్స్, మంత్లీ ఆఫర్స్ ఉంటాయి. వాటిని ఫాలో అవ్వండి. ఫెస్టివల్ సేల్స్ లో కూడా మంచి డిస్కౌంట్స్ దొరుకుతాయి.\n\n4. **బల్క్ లో కొనండి:**  రైస్, పప్పులు, నూనె లాంటివి ఎక్కువ క్వాంటిటీలో కొనడం వల్ల రేట్ తక్కువ పడుతుంది. ఫ్యామిలీ పెద్దది అయితే ఇది చాలా యూస్\u200cఫుల్ టిప్.\n\n5. **సీజనల్ ఫ్రూట్స్ అండ్ వెజిటేబుల్స్ కొనండి:** సీజన్\u200cలో దొరికే కాయగూరలు, పండ్లు

In [6]:
import re
def is_valid_response(response: str) -> bool:
    """Filter out error responses and empty strings."""
    if not response or not response.strip():
        return False
    
    # Filter API errors that appear in IndicVault
    error_patterns = [
        r"ERROR:",
        r"No API keys available",
        r"rate limit",
    ]
    
    for pattern in error_patterns:
        if re.search(pattern, response, re.IGNORECASE):
            return False
    
    return True

In [7]:
dataset = dataset.filter(
    lambda x: is_valid_response(x.get("response", "")),
)

Filter:   0%|          | 0/93028 [00:00<?, ? examples/s]

In [9]:
from tokenizers.processors import TemplateProcessing
from transformers import AutoTokenizer


def get_tokenizer(
    model_name: str = "ai4bharat/IndicBERTv2-MLM-only",
    bos_token: str = "<BOS>",
    eos_token: str = "<EOS>",
    start_token: str = "<START_ID>",
    end_token: str = "<END_ID>",
    eot_token: str = "<EOT_ID>",
):
    """
    Adds repo-style special tokens + chat_template to any MLM tokenizer.

    IMPORTANT DESIGN CHOICE (to avoid double BOS/EOS):
    - We keep TemplateProcessing (BOS ... EOS) enabled for normal tokenization (pretrain script).
    - For chat_template tokenization (SFT + inference prompt), we will call apply_chat_template(add_special_tokens=False)
      so TemplateProcessing does NOT wrap again.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

    special_tokens = {
        "bos_token": bos_token,
        "eos_token": eos_token,
        "additional_special_tokens": [start_token, end_token, eot_token],
    }
    tokenizer.add_special_tokens(special_tokens)

    # PAD/CLS choices like the reference repo
    tokenizer.pad_token = eos_token
    tokenizer.cls_token = bos_token

    # Pretraining post-processor: <BOS> ... <EOS>
    tokenizer._tokenizer.post_processor = TemplateProcessing(
        single=f"{bos_token} $A {eos_token}",
        special_tokens=[
            (bos_token, tokenizer.bos_token_id),
            (eos_token, tokenizer.eos_token_id),
        ],
    )

    # Chat template (same structure as repo)
    tokenizer.chat_template = (
        "{% for message in messages %}"
        "{{ bos_token if loop.first else '' }}"
        f"{{{{ '{start_token}' + message['role'] + '{end_token}' }}}}\n"
        "{{ message['content'] }}"
        f"{{{{ '{eot_token}' if message['role'] == 'user' else eos_token }}}}"
        "{% endfor %}"
        "{% if add_generation_prompt %}"
        f"{{{{ '{start_token}' + 'assistant' + '{end_token}' }}}}"
        "{% endif %}"
    )

    return tokenizer


In [10]:
tokenizer = get_tokenizer("ai4bharat/IndicBERTv2-MLM-only")

In [11]:
def apply_chat_template(query, response):
    """Apply chat template and return list of token IDs."""
    return tokenizer.apply_chat_template(
        [
            {"role": "user", "content": query},
            {"role": "assistant", "content": response}
        ],
        tokenize=True,
        add_special_tokens=True,
    )

In [15]:
dataset[0]

{'question': 'ఇండియాలో గ్రోసరీస్ మీద డబ్బులు సేవ్ చేయడానికి బెస్ట్ వేస్ ఏంటి?\n',
 'response': 'ఇండియాలో గ్రోసరీస్ మీద డబ్బులు సేవ్ చేయడానికి చాలా బెస్ట్ వేస్ ఉన్నాయి. ఇక్కడ కొన్ని టిప్స్ ఉన్నాయి చూడండి:\n\n1. **ముందుగానే ప్లాన్ చేసుకోండి:** షాపింగ్ కి వెళ్ళేముందు ఏం కావాలో లిస్ట్ ప్రిపేర్ చేసుకోండి. లిస్ట్ ఉంటే, అనవసరమైనవి కొనకుండా ఉంటారు, డబ్బులు సేవ్ అవుతాయి.\n\n2. **ప్రైస్ కంపేర్ చేయండి:**  ఒకే ప్రోడక్ట్ వేర్వేరు షాపుల్లో వేర్వేరు రేట్లు ఉండొచ్చు. సూపర్\u200cమార్కెట్స్, లోకల్ షాప్స్, ఆన్\u200cలైన్ స్టోర్స్ లో ప్రైస్ కంపేర్ చేసి ఎక్కడ తక్కువ ఉంటే అక్కడ కొనండి.\n\n3. **సేల్స్ అండ్ డిస్కౌంట్స్ చూడండి:**  సూపర్\u200cమార్కెట్స్ లో వీక్లీ సేల్స్, మంత్లీ ఆఫర్స్ ఉంటాయి. వాటిని ఫాలో అవ్వండి. ఫెస్టివల్ సేల్స్ లో కూడా మంచి డిస్కౌంట్స్ దొరుకుతాయి.\n\n4. **బల్క్ లో కొనండి:**  రైస్, పప్పులు, నూనె లాంటివి ఎక్కువ క్వాంటిటీలో కొనడం వల్ల రేట్ తక్కువ పడుతుంది. ఫ్యామిలీ పెద్దది అయితే ఇది చాలా యూస్\u200cఫుల్ టిప్.\n\n5. **సీజనల్ ఫ్రూట్స్ అండ్ వెజిటేబుల్స్ కొనండి:** సీజన్\u200cలో దొరికే కాయగూరలు, పండ్లు

In [18]:
qna=dataset[0]

In [21]:
question=qna["question"]

In [22]:
response=qna["response"]

In [23]:
chat=apply_chat_template(question, response)

In [24]:
chat

{'input_ids': [250000, 250002, 36153, 250003, 98931, 36805, 9285, 80494, 9230, 15683, 33200, 85272, 219072, 44924, 110349, 103925, 9113, 79264, 78, 250004, 250002, 44788, 250003, 98931, 36805, 9285, 80494, 9230, 15683, 33200, 85272, 219072, 44924, 21165, 110349, 103925, 9113, 23910, 61, 35874, 27066, 34371, 109162, 23910, 71877, 73, 64, 61, 37147, 138399, 71469, 174901, 73, 19946, 196403, 21202, 173406, 66367, 36509, 30169, 189396, 246823, 39048, 30238, 15659, 174901, 61, 246823, 43087, 59, 187763, 152927, 30206, 23313, 58429, 59, 85272, 219072, 172024, 61, 65, 61, 37147, 44268, 15683, 33403, 9288, 15659, 63005, 73, 19946, 47206, 41709, 108301, 9113, 15866, 125240, 181026, 17854, 125240, 158658, 206213, 61, 101523, 24012, 129625, 15683, 59, 79431, 15822, 35136, 109162, 59, 123521, 160391, 15683, 18589, 44268, 15683, 33403, 9288, 15659, 21839, 53453, 40643, 43087, 37490, 30206, 17827, 61, 66, 61, 37147, 219838, 54453, 204092, 15683, 71877, 73, 19946, 101523, 24012, 129625, 15683, 18589,

In [25]:
len(chat)

2

In [29]:
chat["input_ids"]

[250000,
 250002,
 36153,
 250003,
 98931,
 36805,
 9285,
 80494,
 9230,
 15683,
 33200,
 85272,
 219072,
 44924,
 110349,
 103925,
 9113,
 79264,
 78,
 250004,
 250002,
 44788,
 250003,
 98931,
 36805,
 9285,
 80494,
 9230,
 15683,
 33200,
 85272,
 219072,
 44924,
 21165,
 110349,
 103925,
 9113,
 23910,
 61,
 35874,
 27066,
 34371,
 109162,
 23910,
 71877,
 73,
 64,
 61,
 37147,
 138399,
 71469,
 174901,
 73,
 19946,
 196403,
 21202,
 173406,
 66367,
 36509,
 30169,
 189396,
 246823,
 39048,
 30238,
 15659,
 174901,
 61,
 246823,
 43087,
 59,
 187763,
 152927,
 30206,
 23313,
 58429,
 59,
 85272,
 219072,
 172024,
 61,
 65,
 61,
 37147,
 44268,
 15683,
 33403,
 9288,
 15659,
 63005,
 73,
 19946,
 47206,
 41709,
 108301,
 9113,
 15866,
 125240,
 181026,
 17854,
 125240,
 158658,
 206213,
 61,
 101523,
 24012,
 129625,
 15683,
 59,
 79431,
 15822,
 35136,
 109162,
 59,
 123521,
 160391,
 15683,
 18589,
 44268,
 15683,
 33403,
 9288,
 15659,
 21839,
 53453,
 40643,
 43087,
 37490,
 3020

In [30]:
len(chat["input_ids"])

525

In [28]:
chat["attention_mask"]

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,


In [31]:
len(chat["attention_mask"])

525

In [34]:
def get_answer_mask(example):
        """
        Create query_mask where 1 = answer region.
        Exactly matching researcher's implementation.
        """
        tokenized = example["input_ids"]
        
        query_mask = []
        occurrence = 0
        is_answer = False
        
        for t in tokenized:
            check = (t == tokenizer.convert_tokens_to_ids("<END_ID>"))
            
            if not is_answer:
                query_mask.append(0)
            else:
                query_mask.append(1)
            
            if check:
                if occurrence == 0:
                    occurrence += 1
                else:
                    is_answer = True
        
        example["query_mask"] = query_mask
        return example
    
tokenized_data = chat.map(
    get_answer_mask,
)

AttributeError: 